### Scraping the Deputies' information:

In [2]:
import requests
import csv
import pandas as pd
from bs4 import BeautifulSoup
import re

Scrape the main page of the [Chilean Cámara de Diputados](https://www.camara.cl/diputados/diputados.aspx#mostrarDiputados) to extract the names and URLs of each deputy. This will provide a list of political representatives whose speeches we will later analyze.

Visit each deputy's individual page and scrape additional information such as their political party (Partido), district (Distrito), municipality (Comuna), and parliamentary group (Bancada). This metadata will be used to enrich the analysis of their speeches.

Also create the political coalitions:
*   Guberment coalition (Alianza de Gobierno)
*   Other left parties (Partido Humanista)
*   Center (Centro)
*   Center-right wing coalition (Chile Vamos)
*   Right wing (Derecha)

* Independents still have to be classified for coalitions

In [3]:
def clean_name(name):
    prefixes = ["Diputado", "Diputada", "Sr.", "Sra.", "Srta."]
    for prefix in prefixes:
        if prefix in name:
            name = name.replace(prefix, "").strip()
    return name

def normalize_url(url):
    base_url = "https://www.camara.cl"
    if url.startswith("/"):
        url = base_url + url
    elif not url.startswith("http"):
        url = base_url + "/" + url
    if '/diputados/' not in url:
        url = url.replace("/detalle/", "/diputados/detalle/")
    return url

def fetch_page(url):
    try:
        response = requests.get(url)
        response.raise_for_status()
        return BeautifulSoup(response.content, 'html.parser')
    except requests.RequestException as e:
        print(f"Error fetching {url}: {e}")
        return None

def extract_deputy_links(soup):
    return soup.find_all('a', href=lambda x: x and 'detalle/mociones.aspx' in x)

def scrape_deputy_profile(deputy):
    soup = fetch_page(deputy['URL'])
    if not soup:
        return deputy

    name_tag = soup.find('h1', class_='rotulo') or soup.find('h2', class_='rotulo')
    if name_tag:
        deputy['Full Name'] = clean_name(name_tag.text.strip())

    info_div = soup.find('div', class_='grid-3 aleft m-left14', style='margin-right: 20px;')
    if info_div:
        info_text = info_div.get_text(strip=True)

        patterns = {
            'Comunas': r'Comunas:\s*(.*?)(?=Distrito|$)',
            'Distrito': r'Distrito:\s*(.*?)(?=Región|$)',
            'Región': r'Región:\s*(.*?)(?=Período|$)',
            'Período': r'Período:\s*(.*?)(?=Partido|$)',
            'Partido': r'Partido:\s*(.*?)(?=Bancada|$)',
            'Bancada': r'Bancada:\s*(.*?)(?=$)'
        }

        for key, pattern in patterns.items():
            match = re.search(pattern, info_text)
            if match:
                deputy[key] = match.group(1).strip()

    return deputy

def scrape_deputies_main_page(url):
    soup = fetch_page(url)
    if not soup:
        return []

    deputies = extract_deputy_links(soup)
    deputy_list = [{'Name': clean_name(d.text.strip()), 'URL': normalize_url(d['href'])} for d in deputies if d.text.strip()]
    return list({d['URL']: d for d in deputy_list}.values())  # Remove duplicates

def main():
    main_url = "https://www.camara.cl/diputados/diputados.aspx#mostrarDiputados"
    deputies = scrape_deputies_main_page(main_url)
    print(f"Found {len(deputies)} unique deputies")

    with ThreadPoolExecutor(max_workers=10) as executor:
        future_to_deputy = {executor.submit(scrape_deputy_profile, deputy): deputy for deputy in deputies}
        for future in as_completed(future_to_deputy):
            deputy = future_to_deputy[future]
            try:
                data = future.result()
                print(f"Scraped data for: {data.get('Full Name', data['Name'])}")
            except Exception as exc:
                print(f"{deputy['Name']} generated an exception: {exc}")

    df = pd.DataFrame(deputies)

    # Save to CSV
    df.to_csv('chilean_deputies_raw.csv', index=False)
    print("Raw data saved to chilean_deputies_raw.csv")

    return df  # Return the DataFrame

if __name__ == "__main__":
    df_deputies = main()

Found 156 unique deputies


NameError: name 'ThreadPoolExecutor' is not defined